# 03 — DML no Delta Lake (bronze)

Demonstra as 3 operacoes de DML suportadas pelo Delta Lake:
- **INSERT** via `merge` (upsert)
- **UPDATE** condicional
- **DELETE** condicional

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

load_dotenv()

MINIO_ENDPOINT   = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
BRONZE           = os.getenv("MINIO_BRONZE_BUCKET")

spark = (
    SparkSession.builder
    .appName("dml-delta-bronze")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark versao:", spark.version)

## INSERT — novos registros via merge

Insere novos clientes e apolices sem duplicar registros existentes.

In [ ]:
dest_cliente = f"s3a://{BRONZE}/cliente"
dt_cliente   = DeltaTable.forPath(spark, dest_cliente)

print("cliente antes do INSERT:", spark.read.format("delta").load(dest_cliente).count(), "registros")

novos_clientes = spark.createDataFrame([
    (11, "Patricia Nunes",   "111.222.333-44", "1993-07-20", "patricia@email.com", 1),
    (12, "Rafael Oliveira",  "555.666.777-88", "1987-02-14", "rafael@email.com",   2),
], ["id_cliente", "nome", "cpf", "data_nascimento", "email", "id_endereco"])

(
    dt_cliente.alias("t")
    .merge(
        novos_clientes.alias("s"),
        "t.id_cliente = s.id_cliente"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("cliente apos  INSERT:", spark.read.format("delta").load(dest_cliente).count(), "registros")

In [ ]:
dest_sinistro = f"s3a://{BRONZE}/sinistro"
dt_sinistro   = DeltaTable.forPath(spark, dest_sinistro)

print("sinistro antes do INSERT:", spark.read.format("delta").load(dest_sinistro).count(), "registros")

novos_sinistros = spark.createDataFrame([
    (9,  "2024-09-03", "Colisão com animal na pista",  4200.00, "Em análise", 7),
    (10, "2024-10-15", "Granizo danificou lataria",    2800.00, "Em análise", 10),
], ["id_sinistro", "data_ocorrencia", "descricao", "valor_prejuizo", "status", "id_apolice"])

(
    dt_sinistro.alias("t")
    .merge(
        novos_sinistros.alias("s"),
        "t.id_sinistro = s.id_sinistro"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

print("sinistro apos  INSERT:", spark.read.format("delta").load(dest_sinistro).count(), "registros")

## UPDATE — atualizacao condicional

In [ ]:
dest_apolice = f"s3a://{BRONZE}/apolice"
dt_apolice   = DeltaTable.forPath(spark, dest_apolice)

print("apolice ANTES do UPDATE (cobertura Basica):")
spark.read.format("delta").load(dest_apolice).filter(col("cobertura") == "Básica").show(truncate=False)

# Reajuste de 15% no prêmio das apólices básicas
dt_apolice.update(
    condition=col("cobertura") == "Básica",
    set={"valor_premio": col("valor_premio") * lit(1.15)}
)

print("apolice APOS UPDATE (reajuste 15% nas Basicas):")
spark.read.format("delta").load(dest_apolice).filter(col("cobertura") == "Básica").show(truncate=False)

In [ ]:
# Atualiza status de sinistros em analise para Aprovado
print("sinistro ANTES do UPDATE:")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

dt_sinistro.update(
    condition=col("status") == "Em análise",
    set={"status": lit("Aprovado")}
)

print("sinistro APOS UPDATE:")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

## DELETE — exclusao condicional

In [ ]:
print(f"sinistro ANTES do DELETE: {spark.read.format('delta').load(dest_sinistro).count()} registros")

# Remove sinistros com status Recusado
dt_sinistro.delete(condition=col("status") == "Recusado")

print(f"sinistro APOS  DELETE: {spark.read.format('delta').load(dest_sinistro).count()} registros")
spark.read.format("delta").load(dest_sinistro).show(truncate=False)

In [ ]:
dest_carro = f"s3a://{BRONZE}/carro"
dt_carro   = DeltaTable.forPath(spark, dest_carro)

print(f"carro ANTES do DELETE: {spark.read.format('delta').load(dest_carro).count()} registros")

# Remove carros fabricados antes de 2018
dt_carro.delete(condition=col("ano") < lit(2018))

print(f"carro APOS  DELETE: {spark.read.format('delta').load(dest_carro).count()} registros")

## Historico de versoes (Time Travel)

In [ ]:
for table in ["cliente", "apolice", "sinistro", "carro"]:
    dest    = f"s3a://{BRONZE}/{table}"
    dt      = DeltaTable.forPath(spark, dest)
    history = dt.history().select("version", "timestamp", "operation")
    print(f"{table}:")
    history.show(truncate=False)